## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/DC_Guardian/ppe_detection")
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
MODELS = PROJECT / "models"
RESULTS = PROJECT / "results"
SRC = PROJECT / "src"

for folder in [RAW, PROCESSED, MODELS, RESULTS, SRC]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)


Mounted at /content/drive
Project: /content/drive/MyDrive/DC_Guardian/ppe_detection


## 2. Install dependencies

In [2]:
!pip install -q ultralytics kaggle pyyaml

import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable a GPU runtime in Colab before training.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 5.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cpu
Ultralytics: 8.4.162
CUDA available: False
Enable a GPU runtime in Colab before training.


## 3. Kaggle authentication
Download `kaggle.json` from your Kaggle account settings, then run this cell and upload it.

Do not publish or commit `kaggle.json`.


In [ ]:
from google.colab import files
import os

uploaded = files.upload()

if "kaggle.json" not in uploaded:
    raise FileNotFoundError("Please upload kaggle.json")

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(uploaded["kaggle.json"])

os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials configured.")


## 4. Download the SH17 dataset

In [ ]:
import subprocess

dataset_id = "mugheesahmad/sh17-dataset-for-ppe-detection"

subprocess.run([
    "kaggle", "datasets", "download",
    "-d", dataset_id,
    "-p", str(RAW),
    "--force"
], check=True)

print("Downloaded files:")
for p in RAW.iterdir():
    print(" -", p.name)


## 5. Extract SH17

In [ ]:
import zipfile

zip_files = list(RAW.glob("*.zip"))
if not zip_files:
    raise FileNotFoundError("No ZIP file was found in the raw dataset directory.")

extract_dir = RAW / "SH17"
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_files[0], "r") as z:
    z.extractall(extract_dir)

print("Extracted to:", extract_dir)


## 6. Inspect dataset structure

In [ ]:
import os

dataset_root = extract_dir

for root, dirs, files in os.walk(dataset_root):
    level = len(Path(root).relative_to(dataset_root).parts)
    print("    " * level + Path(root).name + "/")
    for name in files[:8]:
        print("    " * (level + 1) + name)
    if level >= 3:
        dirs[:] = []


## 7. Find and inspect YAML configuration

In [ ]:
import yaml

yaml_files = list(dataset_root.rglob("*.yaml")) + list(dataset_root.rglob("*.yml"))

print("YAML files:")
for p in yaml_files:
    print(" -", p)

dataset_yaml = None

for p in yaml_files:
    try:
        data = yaml.safe_load(p.read_text())
        if isinstance(data, dict) and "names" in data:
            dataset_yaml = p
            print("\nSelected:", dataset_yaml)
            print(yaml.safe_dump(data, sort_keys=False))
            break
    except Exception:
        pass

if dataset_yaml is None:
    print("\nNo YOLO dataset YAML was automatically identified.")


## 8. Count images and annotation files

In [ ]:
from collections import Counter

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
images = [p for p in dataset_root.rglob("*") if p.suffix.lower() in image_exts]
labels = list(dataset_root.rglob("*.txt"))

print("Images:", len(images))
print("TXT files:", len(labels))

print("\nSample images:")
for p in images[:5]:
    print(p)

print("\nSample TXT files:")
for p in labels[:5]:
    print(p)


## 9. Inspect YOLO annotation class IDs

In [ ]:
class_counts = Counter()
valid_yolo_rows = 0

for label_file in labels:
    try:
        for line in label_file.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                try:
                    class_id = int(float(parts[0]))
                    coords = list(map(float, parts[1:]))
                    if all(0 <= x <= 1 for x in coords):
                        class_counts[class_id] += 1
                        valid_yolo_rows += 1
                except ValueError:
                    pass
    except Exception:
        pass

print("Valid YOLO-style annotation rows:", valid_yolo_rows)
print("\nClass distribution:")
for class_id, count in sorted(class_counts.items()):
    print(f"Class {class_id}: {count}")


## 10. Display sample images

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

if not images:
    raise RuntimeError("No images were found.")

sample_images = random.sample(images, min(6, len(images)))

plt.figure(figsize=(15, 10))
for i, image_path in enumerate(sample_images):
    img = Image.open(image_path).convert("RGB")
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(image_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()


## 11. Review checkpoint

Before training, confirm:

- the exact SH17 classes;
- the annotation format;
- that bounding boxes/classes are sensible;
- train/validation/test structure;
- which PPE classes will be required by the prototype.

The project guide requires this inspection before finalizing the compliance policy.


## 12. Prepare a fast local training copy

Google Drive is used for permanent storage. Colab's `/content` disk is preferable for training speed.

This cell copies the extracted dataset locally. It does not modify the raw Drive copy.


In [ ]:
import shutil

LOCAL_DATASET = Path("/content/SH17")

if LOCAL_DATASET.exists():
    shutil.rmtree(LOCAL_DATASET)

shutil.copytree(dataset_root, LOCAL_DATASET)

print("Local training copy:", LOCAL_DATASET)


## 13. Locate local `data.yaml`

Run this after verifying that the detected YAML is the correct YOLO dataset configuration.


In [ ]:
local_yaml_files = list(LOCAL_DATASET.rglob("*.yaml")) + list(LOCAL_DATASET.rglob("*.yml"))

for p in local_yaml_files:
    print(p)

print("\nSet DATA_YAML below to the correct file before training.")


## 14. Set the training YAML

In [ ]:
# CHANGE THIS if your inspection above shows a different path.
# Example:
# DATA_YAML = "/content/SH17/data.yaml"

DATA_YAML = None

print("DATA_YAML =", DATA_YAML)


## 15. Train YOLOv8

This cell deliberately refuses to start until `DATA_YAML` has been set to the verified SH17 YOLO configuration.

Start with YOLOv8n for faster experimentation.


In [ ]:
from ultralytics import YOLO

if DATA_YAML is None:
    raise ValueError("Set DATA_YAML in the previous cell after inspecting SH17.")

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/content/ppe_runs",
    name="sh17_yolov8n",
    seed=42,
    pretrained=True
)


## 16. Save the best checkpoint to Google Drive

In [ ]:
import shutil

best_model = Path("/content/ppe_runs/sh17_yolov8n/weights/best.pt")

if not best_model.exists():
    raise FileNotFoundError(f"Best model not found: {best_model}")

drive_best = MODELS / "best.pt"
shutil.copy2(best_model, drive_best)

print("Saved:", drive_best)


## 17. Evaluate the trained model

In [ ]:
trained_model = YOLO(str(MODELS / "best.pt"))

metrics = trained_model.val(data=DATA_YAML)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Mean precision:", metrics.box.mp)
print("Mean recall:", metrics.box.mr)


## 18. Test inference on an image

In [ ]:
from google.colab import files

test_upload = files.upload()
test_image = next(iter(test_upload.keys()))

predictions = trained_model.predict(
    source=test_image,
    conf=0.25,
    save=True
)

print("Inference complete.")


## 19. Generic structured detection output

This first wrapper reports detected objects without inventing a compliance policy. After SH17's class mapping is confirmed, extend this into the final `infer_ppe(frame)` COMPLIANT/NON_COMPLIANT wrapper.


In [ ]:
def detect_ppe(frame, model=trained_model, conf=0.25):
    result = model.predict(frame, conf=conf, verbose=False)[0]
    names = result.names

    detections = []

    if result.boxes is not None:
        for box in result.boxes:
            cls_id = int(box.cls.item())
            confidence = float(box.conf.item())
            xyxy = [float(v) for v in box.xyxy[0].tolist()]

            detections.append({
                "class_id": cls_id,
                "class_name": names[cls_id],
                "confidence": round(confidence, 4),
                "bbox_xyxy": xyxy
            })

    return {
        "detections": detections,
        "count": len(detections),
        "model": "YOLOv8 SH17"
    }

# Example:
# output = detect_ppe(test_image)
# print(output)


# Next development step

After the dataset inspection establishes the exact SH17 class names, implement:

`infer_ppe(frame) -> detected PPE + confidence + COMPLIANT/NON_COMPLIANT`

For multiple workers, the compliance logic should associate PPE with individual person detections rather than treating PPE anywhere in the frame as belonging to every worker.
